# Structured Web Data Extraction Agent
This agent pipeline handles extracting complex thread structures from web pages using LangGraph and Tavily web research.

In [1]:
import os
import json
import warnings
from dotenv import load_dotenv

warnings.filterwarnings('ignore')
load_dotenv()

from typing import Annotated, Literal, Optional, List, Dict, Any
from langchain_core.messages import HumanMessage
from langgraph.graph import MessagesState, START, StateGraph, END
from langgraph.types import Command
from langgraph.prebuilt import create_react_agent
from langchain_groq import ChatGroq
from langchain_community.tools.tavily_search import TavilySearchResults

import prompts

# Setup LLMs
# Use the fast routing model for the agent node
reasoning_llm = ChatGroq(model='llama-3.1-8b-instant')
llm = reasoning_llm

# Specialized State for Data Extracting Pipeline
class State(MessagesState):
    enabled_agents: Optional[List[str]]
    plan: Optional[Dict[str, Dict[str, Any]]]
    user_query: Optional[str]
    current_step: int
    replan_flag: Optional[bool]
    last_reason: Optional[str]
    replan_attempts: Optional[Dict[int, int]]
    agent_query: Optional[str]
    final_answer: Optional[str]

MAX_REPLANS = 2


In [2]:
# Agent Setup
tavily_tool = TavilySearchResults(
    max_results=5,
    search_depth='advanced',
    include_raw_content=True
)

web_search_agent = create_react_agent(
    llm,
    tools=[tavily_tool],
    prompt=prompts.agent_system_prompt(
        'You are the Web Data Extraction Researcher. Use the provided search tool to thoroughly search the given URL or topic. '
        'You must thoroughly extract the main points, user comments, and insights. Your output must be highly detailed so the Synthesizer can present a comprehensive final report.\n'
        'IMPORTANT: Provide ONLY the JSON arguments for tools. Do not wrap the tool call in XML tags or markdown. Do NOT use <function=...> tags.'
    ),
)

C:\Users\Sanjaya\AppData\Local\Temp\ipykernel_25780\1692858415.py:2: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tavily_tool = TavilySearchResults(


In [3]:
# Core Node Definitions
import re
import requests

def planner_node(state: State) -> Command[Literal['executor']]:
    # --- UPGRADED PLANNER PROMPT WITH PRE/POST CONDITIONS ---
    # According to best practices, adding pre-conditions, post-conditions,
    # and a explicit goal improves the execution efficiency and logical consistency.
    from langchain_core.messages import HumanMessage
    
    plan_instructions = """You are the Planner. Determine a clear plan to fulfill the user query.
Generate a JSON output mapping each step "1", "2", etc., to an object with:
"agent": (web_researcher or synthesizer)
"pre_conditions": [list of strings]
"post_conditions": [list of strings]
"goal": "string"

User Query: """ + state.get("user_query", "")

    llm_reply = reasoning_llm.invoke([HumanMessage(content=plan_instructions)])
    
    try:
        content_str = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
        content_str = content_str.replace('```json', '').replace('```', '')
        parsed_plan = json.loads(content_str)
    except Exception as e:
        print(f"Planning failed/fallback used: {e}")
        parsed_plan = {
            "1": {
                "agent": "web_researcher", 
                "goal": "Extract all available Reddit text data.",
                "pre_conditions": ["Have a valid Reddit URL"],
                "post_conditions": ["A detailed breakdown of the post and ALL comments exists in state"]
            },
            "2": {
                "agent": "synthesizer", 
                "goal": "Summarize the extracted data securely into structured sections.",
                "pre_conditions": ["Extracted Reddit data is available"],
                "post_conditions": ["A comprehensive final report is generated without individual commenter names"]
            }
        }

    return Command(
        update={
            "plan": parsed_plan,
            "messages": [HumanMessage(content=json.dumps(parsed_plan), name="initial_plan")],
            "user_query": state.get("user_query", state.get("messages", [{}])[0].content if state.get("messages") else ""),
            "current_step": 1,
            "replan_flag": False,
            "last_reason": "",
            "enabled_agents": state.get("enabled_agents"),
        },
        goto="executor",
    )

def executor_node(state: State) -> Command[Literal["web_researcher", "synthesizer", "planner"]]:
    plan = state.get("plan", {})
    step = state.get("current_step", 1)
    
    planned_agent = plan.get(str(step), {}).get("agent", "synthesizer")
    
    valid_agents = ["web_researcher", "synthesizer", "planner"]
    if planned_agent not in valid_agents:
        planned_agent = "synthesizer"

    updates = {
        "messages": [HumanMessage(content=f"Routing execution to {planned_agent} for step {step}", name="executor")],
        "last_reason": "strict plan adherence",
        "current_step": step + 1,
        "replan_flag": False
    }

    return Command(update=updates, goto=planned_agent)


# --- UPGRADED WEB RESEARCHER TO EXTRACT ALL COMMENTS INSTEAD OF TOP 15 ---
def web_research_node(state: State) -> Command[Literal["executor"]]:
    query = state.get("user_query", "")
    print(f"WEB SEARCH POINTER -> Querying: {query[:50]}...")
    
    final_content = ""
    # Extract the exact URL to check if it's a reddit URL
    url_match = re.search(r"https?://[^\s]+", query)
    target_url = url_match.group(0) if url_match else None
    
    if target_url and "reddit.com" in target_url:
        try:
            # We add ?limit=1000 to the JSON URL to request up to 1000 comments (Reddit API max limit)
            json_url = target_url.rstrip("/") + ".json?limit=1000"
            headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'}
            
            response = requests.get(json_url, headers=headers)
            if response.status_code == 200:
                data = response.json()
                post_data = data[0]['data']['children'][0]['data']
                title = post_data.get('title', '')
                selftext = post_data.get('selftext', '')
                
                # Fetch ALL available top level comments
                comments = []
                if len(data) > 1:
                    comment_children = data[1]['data'].get('children', [])
                    for child in comment_children:
                        if child.get('kind') == 't1':  # 't1' indicates a comment
                            c_data = child['data']
                            body = c_data.get('body', '').strip()
                            if body and body != "[deleted]" and body != "[removed]":
                                comments.append(f"Comment (Score: {c_data.get('score', 0)}):\n{body}")
                
                # Format to a large string. For massive threads, we truncate to safely fit 
                # context window limits, but it safely covers far more than just 15
                combined_comments = "\n---\n".join(comments)
                
                # Limit size to ~60,000 chars roughly to avoid groq token limit overload
                if len(combined_comments) > 60000:
                    combined_comments = combined_comments[:60000] + "\n...[Additional Comments Truncated Due To Context Size]..."
                    
                final_content = f"Extracted from Reddit URL:\n\nTitle: {title}\n\nPost Content:\n{selftext}\n\nALL Extracted Comments ({len(comments)} total top-level comments):\n{combined_comments}"
                print(f"✅ Successfully scraped Post + {len(comments)} available top-level comments.")
        except Exception as e:
            print(f"Reddit JSON fetch failed: {e}")
    
    # If it wasn't a reddit URL or the direct fetch failed, fallback to the agent
    if not final_content:
        try:
            agent_result = web_search_agent.invoke({"messages": [HumanMessage(content=query)]})
            agent_messages = agent_result.get("messages", [])
            final_content = agent_messages[-1].content if agent_messages else "No output from agent."
        except Exception as e:
            final_content = f"Web search agent failed: {e}"
        
    result_message = HumanMessage(content=final_content, name="web_researcher")
    
    state_messages = state.get("messages", []) + [result_message]
    return Command(update={"messages": state_messages}, goto="executor")

def synthesizer_node(state: State) -> Command[Literal[END]]:
    relevant_msgs = [m.content for m in state.get("messages", []) if getattr(m, "name", None) in ("web_researcher",)]
    user_question = state.get("user_query", state.get("messages", [{}])[0].content if state.get("messages") else "")
    
    system_instructions = (
        "Task: Analyze the provided Reddit extraction.\n"
        "You MUST format your response into the following distinct sections:\n\n"
        "1. **Post Content Summary**: A highly detailed summary of the main original post.\n"
        "2. **Comments & Community Reaction**: A synthesized summary of the ENTIRE community's response. Extract the overarching themes, agreements, disagreements, and main arguments based from ALL the provided comments. MUST NOT mention individual commenter names or directly quote them with their names.\n"
        "3. **Key Insights & Takeaways**: Any final important patterns, tips, or lessons learned from the thread.\n\n"
        "Ensure extreme detail based ONLY on the context gathered below."
    )
    
    prompt = f"User question: {user_question}\n\n{system_instructions}\n\nContext:\n\n---\n" + "\n\n---\n".join(relevant_msgs)
    
    # Increase truncation check so we don't accidentally cut out the long comments section we just grabbed
    if len(prompt) > 80000:
       prompt = prompt[:80000]
       
    heavy_llm = ChatGroq(model="llama-3.1-8b-instant")
    llm_reply = heavy_llm.invoke([HumanMessage(content=prompt)])
    answer = llm_reply.content if isinstance(llm_reply.content, str) else str(llm_reply.content)
    return Command(update={"final_answer": answer.strip(), "messages": [HumanMessage(content=answer.strip(), name="synthesizer")]}, goto=END)

In [4]:
from langgraph.checkpoint.memory import MemorySaver

# Build Data Extraction Graph with An Interrupt Pause
workflow = StateGraph(State)
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("web_researcher", web_research_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "planner")

# Add a checkpointer so we can pause before the final synthesis step
memory = MemorySaver()
graph = workflow.compile(checkpointer=memory, interrupt_before=["synthesizer"])

In [5]:
# Define the exact extraction target query
target_url = "https://www.reddit.com/r/ClaudeCode/comments/1r5gk1d/40_days_of_vibe_coding_taught_me_the_most/"

query = f"""Please thoroughly extract the main points, insights, and key takeaways from the following Reddit post URL:
{target_url}

Also, summarize any notable comments or community reactions discussed below the post. Provide a heavily detailed breakdown so I can manually verify the facts."""

state = {
    "messages": [HumanMessage(content=query)],
    "user_query": query,
    "enabled_agents": ["web_researcher", "synthesizer"],
}

config = {"configurable": {"thread_id": "reddit_research_thread"}}

print("Starting Web Extraction Pipeline... (Will pause before Synthesizer)\n")
try:
    # This will run up to just BEFORE the 'synthesizer' node since we set interrupt_before
    result = graph.invoke(state, config=config)
    
    # We retrieve the state messages from the web_researcher to display to you
    state_history = graph.get_state(config)
    messages = state_history.values.get("messages", [])
    
    web_research_results = [msg.content for msg in messages if getattr(msg, "name", "") == "web_researcher"]
    
    if web_research_results:
        print("--- RAW WEB RESEARCHER EXTRACTED CONTENT ---")
        print(web_research_results[-1])
        print("\n-----------------------------------------------\n")
        print("✅ PAUSED: Review the scraped data above. If it's correct, run the next cell to trigger the Synthesizer!")
    else:
        print("Web researcher hasn't produced output yet.")
        
except Exception as e:
    print(f"Pipeline execution failed: {e}")

Starting Web Extraction Pipeline... (Will pause before Synthesizer)

Planning failed/fallback used: Expecting value: line 1 column 1 (char 0)
WEB SEARCH POINTER -> Querying: Please thoroughly extract the main points, insight...
✅ Successfully scraped Post + 52 available top-level comments.
--- RAW WEB RESEARCHER EXTRACTED CONTENT ---
Extracted from Reddit URL:

Title: 40 days of vibe coding taught me the most important skill isn't prompting. It's something way more boring.

Post Content:
Been building a developer tool for internal business apps entirely with Claude Code for the last 40 days. Not a weekend project - full stack with auth, RBAC, API layer, data tables, email system, S3 support, PostgreSQL local and cloud. No hand-written code - I describe what I want, review output, iterate.

Yesterday I ran a deep dive on my git history because I wanted to understand what actually happened over those 40 days. 312 commits, 36K lines of code, 176 components, 53 API endpoints.

And the thin

In [6]:
# Run this cell to resume the pipeline and synthesize the findings
print("Resuming Pipeline for Synthesis Phase...\n")
try:
    # Passing None as the input resumes the graph with the exact same thread_id config
    final_result = graph.invoke(None, config=config)
    
    print("--- 📝 FINAL SYNTHESIZED REPORT ---\n")
    print(final_result.get("final_answer", final_result))
    
except Exception as e:
    print(f"Synthesis failed: {e}")

Resuming Pipeline for Synthesis Phase...

--- 📝 FINAL SYNTHESIZED REPORT ---

**1. Post Content Summary:**

The original post discusses the author's 40-day experience using Claude Code to build a full-stack developer tool entirely with code generated by the AI. The author highlights the importance of a constraints document, specifically the `CLAUDE.md` file, which defines the architecture, patterns, and naming conventions for the project. The author reveals that this file was the single most edited file in the project, with 43 changes, surpassing the number of changes to any React component or API route.

The author argues that the most important skill in working with AI-generated code is not prompting, but rather defining the constraints and architecture of the project. They claim that this approach has led to a better feature-to-fix ratio, a larger codebase with no complexity wall, and a flat bug fix frequency. The author invites the community to share their thoughts on whether other